# Aufgabe 2: Analyse der Bilddynamik

## Ziel
Die extrahierten Frames sollen analysiert werden, um die "Dynamik" des Videos zu quantifizieren.  
Dabei werden zwei Haupt-Features berechnet:

1. **schnitt_frequenz** – wie oft wechselt die Szene  
2. **durchschnittliche_bewegung** – wie viel Bewegung im Video vorhanden ist

---

## Zu erstellende Features

| Feature | Beschreibung | Methode |
|----------|---------------|----------|
| **schnitt_frequenz** | Anzahl der Szenenwechsel im Video | Scene Change Detection über Histogramm-Vergleich |
| **durchschnittliche_bewegung** | Durchschnittliche Bewegung zwischen aufeinanderfolgenden Frames | Optical Flow Analyse |

---

## Algorithmen

### Schnittfrequenz
Zur Erkennung von Szenenwechseln wird das Histogramm jedes Frames berechnet und mit dem vorherigen verglichen.  
Ein deutlicher Unterschied im Histogramm weist auf einen Schnitt hin.

**Verwendete Methode:**
```python
cv2.compareHist(hist1, hist2, cv2.HISTCMP_BHATTACHARYYA)
```

## 1. Setup & Import

In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
from collections import defaultdict
import math
import re # Regular Expressions für robustere Namenserkennung

# --- Konfiguration ---

# 1. Eingabe: Der Ordner mit den extrahierten Frames (aus Skript 1)
FRAME_DIR = "data/processed/video_frames"

# 2. Ausgabe: Die Datei, in der wir alle Video-Features speichern
FEATURE_FILE = "features/video_features.csv"

# 3. Parameter 
# Schwellenwert für Histogramm-Vergleich (0 = identisch, 1 = völlig verschieden)
# Ein Wert von 0.3 bedeutet, das Bild muss sich zu >30% ändern, um als "Schnitt" zu gelten.
SCENE_CHANGE_THRESHOLD = 0.3

# FRAMES_PER_SECOND_TO_SAVE muss mit dem Wert aus dem
# *vorherigen* Notebook (T3_01) übereinstimmen!
FRAMES_PER_SECOND_TO_SAVE = 1

## 2. Definition der Analyse-Funktionen

In [7]:
def calculate_scene_changes(sorted_frames_list, threshold):
    """
    Analysiert eine Liste von geladenen Frames und zählt die Schnitte.
    """
    if len(sorted_frames_list) < 2:
        return 0
    
    scene_changes = 0
    
    # Erstes Frame als Referenz nehmen
    prev_frame = sorted_frames_list[0]
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    prev_hist = cv2.calcHist([prev_gray], [0], None, [256], [0, 256])
    cv2.normalize(prev_hist, prev_hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
    
    for i in range(1, len(sorted_frames_list)):
        current_frame = sorted_frames_list[i]
        current_gray = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY)
        current_hist = cv2.calcHist([current_gray], [0], None, [256], [0, 256])
        cv2.normalize(current_hist, current_hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
        
        # Vergleiche das Histogramm des aktuellen Frames mit dem vorherigen
        distance = cv2.compareHist(prev_hist, current_hist, cv2.HISTCMP_BHATTACHARYYA)
        
        if distance > threshold:
            scene_changes += 1
            
        # Aktuelles Frame wird zum "vorherigen" Frame für die nächste Iteration
        prev_hist = current_hist
        
    return scene_changes

def calculate_average_motion(sorted_frames_list):
    """
    Analysiert eine Liste von geladenen Frames und berechnet die durchschnittliche Bewegung.
    """
    if len(sorted_frames_list) < 2:
        return 0
    
    all_magnitudes = []
    
    prev_gray = cv2.cvtColor(sorted_frames_list[0], cv2.COLOR_BGR2GRAY)
    
    for i in range(1, len(sorted_frames_list)):
        current_gray = cv2.cvtColor(sorted_frames_list[i], cv2.COLOR_BGR2GRAY)
        
        # Berechne den "Optical Flow"
        flow = cv2.calcOpticalFlowFarneback(prev_gray, current_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        # Berechne die Stärke (Magnitude) der Bewegung
        magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        
        # Speichere den Durchschnittswert der Bewegung für dieses Frame-Paar
        all_magnitudes.append(np.mean(magnitude))
        
        prev_gray = current_gray
        
    if not all_magnitudes:
        return 0
    
    # Gib den Durchschnitt über das gesamte Video zurück
    return np.mean(all_magnitudes)

# Hypothese

Die Annahme ist, dass **virale Videos (Top_100)** eine **signifikant höhere visuelle Dynamik** aufweisen als **normale Videos (Normal_100)**.  

Die **Zuschauerbindung ("Attention")** wird durch **schnelle Schnitte** und **hohe Bewegungsintensität** verstärkt.

---

## Erwartung
Es wird daher erwartet, dass:

- die **schnitt_frequenz**  
  (Anzahl der Szenenwechsel)  
  bei viralen Videos im Durchschnitt **höher** ist.

- die **durchschnittliche_bewegung**  
  (gemessen über den optischen Fluss)  
  bei viralen Videos ebenfalls **höher** ausfällt.

## 3. Hauptverarbeitung
Die Tausenden von Frames müssen wieder den ursprünglichen Videos zuordnen und dann die Analysefunktionen auf jede Gruppe anwenden.

In [8]:
# 1. Alle Frame-Dateien auflisten
try:
    all_frame_files = os.listdir(FRAME_DIR)
    jpg_files = [f for f in all_frame_files if f.endswith('.jpg')]
    if not jpg_files:
        print(f"FEHLER: Keine .jpg-Dateien in {FRAME_DIR} gefunden.")
        print("Stelle sicher, dass Skript T3_01 (Frame Extraction) erfolgreich war.")
    else:
        print(f"Insgesamt {len(jpg_files)} Frames gefunden.")
except FileNotFoundError:
    print(f"FEHLER: Verzeichnis nicht gefunden: {FRAME_DIR}")
    jpg_files = []

# 2. Frames pro Video gruppieren (ROBUSTERE METHODE)
video_groups = defaultdict(list)
frame_name_pattern = re.compile(r"^(.*?)_frame_(\d+)\.jpg$") # z.B. (top_95_...)_frame_(30).jpg

for f in jpg_files:
    match = frame_name_pattern.match(f)
    if match:
        video_name = match.group(1) # Das ist der "video_id"
        frame_number = int(match.group(2)) # Das ist die Frame-Nummer
        video_groups[video_name].append((frame_number, f)) # Speichere (Nummer, Dateiname)
    else:
        print(f"WARNUNG: Datei {f} passt nicht zum erwarteten Muster und wird ignoriert.")

print(f"{len(video_groups)} einzigartige Videos identifiziert.")
if len(video_groups) == 0 and len(jpg_files) > 0:
    print("FEHLER: Frames wurden gefunden, aber konnten keinem Video zugeordnet werden.")
    print("Bitte prüfe das Namensmuster (z.B. 'video_name_frame_123.jpg')")

# 3. Features extrahieren (Dieser Schritt kann dauern!)
results = []
os.makedirs("features", exist_ok=True) 

for video_name, frame_list in video_groups.items():
    print(f"\nVerarbeite Video: {video_name} ({len(frame_list)} Frames)")
    
    # 3a. Frames nach ihrer Nummer sortieren
    frame_list.sort(key=lambda x: x[0])
    
    # 3b. Frames laden
    loaded_frames = []
    for frame_num, frame_file in frame_list:
        frame_path = os.path.join(FRAME_DIR, frame_file)
        img = cv2.imread(frame_path)
        if img is not None:
            loaded_frames.append(img)
            
    if len(loaded_frames) < 2:
        print(f"  INFO: Zu wenige Frames für {video_name}, um Analyse durchzuführen.")
        continue

    # 3c. Features berechnen
    scene_changes = calculate_scene_changes(loaded_frames, SCENE_CHANGE_THRESHOLD)
    avg_motion = calculate_average_motion(loaded_frames)
    
    # 3d. Schnittfrequenz berechnen
    video_duration_seconds = len(loaded_frames) / FRAMES_PER_SECOND_TO_SAVE
    schnitt_frequenz = scene_changes / video_duration_seconds if video_duration_seconds > 0 else 0
    
    print(f"  -> Schnitte: {scene_changes} (Frequenz: {schnitt_frequenz:.2f})")
    print(f"  -> Bewegung: {avg_motion:.2f}")

    # 3e. Ergebnis speichern
    results.append({
        "video_id": video_name,
        "schnitt_frequenz": schnitt_frequenz,
        "durchschnittliche_bewegung": avg_motion,
        "anzahl_frames": len(loaded_frames),
        "video_dauer_sek": video_duration_seconds
    })

print("\n--- Feature-Extraktion abgeschlossen ---")

Insgesamt 6789 Frames gefunden.
197 einzigartige Videos identifiziert.

Verarbeite Video: top_53_likes_729700_id_7542648831586880823 (107 Frames)
  -> Schnitte: 23 (Frequenz: 0.21)
  -> Bewegung: 9.15

Verarbeite Video: top_45_likes_780200_id_7548598130665622804 (26 Frames)
  -> Schnitte: 5 (Frequenz: 0.19)
  -> Bewegung: 6.86

Verarbeite Video: top_96_likes_365300_id_7560113050100043026 (55 Frames)
  -> Schnitte: 16 (Frequenz: 0.29)
  -> Bewegung: 5.53

Verarbeite Video: top_32_likes_1000000_id_7556001068405050638 (253 Frames)
  -> Schnitte: 28 (Frequenz: 0.11)
  -> Bewegung: 9.76

Verarbeite Video: top_19_likes_1500000_id_7231352152743152942 (29 Frames)
  -> Schnitte: 0 (Frequenz: 0.00)
  -> Bewegung: 3.91

Verarbeite Video: normal_11_likes_30500_id_7559969376590515470 (29 Frames)
  -> Schnitte: 10 (Frequenz: 0.34)
  -> Bewegung: 4.97

Verarbeite Video: normal_2_likes_58100_id_7525148037694393614 (33 Frames)
  -> Schnitte: 7 (Frequenz: 0.21)
  -> Bewegung: 5.64

Verarbeite Video: nor

In [11]:
# 3. Features extrahieren (Dieser Schritt kann dauern!)

results = []
os.makedirs("features", exist_ok=True) # Stellt sicher, dass der Ordner "features" existiert

# Iteriere durch jedes Video (jede Gruppe von Frames)
for video_name, frame_files in video_groups.items():
    
    print(f"\nVerarbeite Video: {video_name} ({len(frame_files)} Frames)")
    
    # 3a. Frames numerisch sortieren
    # Wichtig, damit frame_30 nach frame_0 kommt!
    try:
        frame_files.sort(key=lambda x: x[0])  # Sortiere nach frame_number (erstes Element des Tupels)
    except Exception as e:
        print(f"  WARNUNG: Konnte Frames für {video_name} nicht sortieren. Überspringe... ({e})")
        continue

    # 3b. Frames laden
    loaded_frames = []
    for frame_num, frame_file in frame_files:  # Tupel entpacken!
        frame_path = os.path.join(FRAME_DIR, frame_file)
        img = cv2.imread(frame_path)
        if img is not None:
            loaded_frames.append(img)
        else:
            print(f"  WARNUNG: Konnte Frame {frame_file} nicht laden.")
            
    if len(loaded_frames) < 2:
        print(f"  INFO: Zu wenige Frames für {video_name}, um Analyse durchzuführen.")
        continue

    # 3c. Features berechnen
    scene_changes = calculate_scene_changes(loaded_frames, SCENE_CHANGE_THRESHOLD)
    avg_motion = calculate_average_motion(loaded_frames)
    
    # 3d. Schnittfrequenz berechnen
    # (Anzahl der Schnitte pro Sekunde Video)
    video_duration_seconds = len(loaded_frames) / FRAMES_PER_SECOND_TO_SAVE
    schnitt_frequenz = scene_changes / video_duration_seconds if video_duration_seconds > 0 else 0
    
    print(f"  -> Schnitte: {scene_changes} (Frequenz: {schnitt_frequenz:.2f})")
    print(f"  -> Bewegung: {avg_motion:.2f}")

    # 3e. Ergebnis speichern
    results.append({
        "video_id": video_name,
        "schnitt_frequenz": schnitt_frequenz,
        "durchschnittliche_bewegung": avg_motion,
        "anzahl_frames": len(loaded_frames),
        "video_dauer_sek": video_duration_seconds
    })

print("\n--- Feature-Extraktion abgeschlossen ---")


Verarbeite Video: top_53_likes_729700_id_7542648831586880823 (107 Frames)
  -> Schnitte: 23 (Frequenz: 0.21)
  -> Bewegung: 9.15

Verarbeite Video: top_45_likes_780200_id_7548598130665622804 (26 Frames)
  -> Schnitte: 5 (Frequenz: 0.19)
  -> Bewegung: 6.86

Verarbeite Video: top_96_likes_365300_id_7560113050100043026 (55 Frames)
  -> Schnitte: 16 (Frequenz: 0.29)
  -> Bewegung: 5.53

Verarbeite Video: top_32_likes_1000000_id_7556001068405050638 (253 Frames)
  -> Schnitte: 28 (Frequenz: 0.11)
  -> Bewegung: 9.76

Verarbeite Video: top_19_likes_1500000_id_7231352152743152942 (29 Frames)
  -> Schnitte: 0 (Frequenz: 0.00)
  -> Bewegung: 3.91

Verarbeite Video: normal_11_likes_30500_id_7559969376590515470 (29 Frames)
  -> Schnitte: 10 (Frequenz: 0.34)
  -> Bewegung: 4.97

Verarbeite Video: normal_2_likes_58100_id_7525148037694393614 (33 Frames)
  -> Schnitte: 7 (Frequenz: 0.21)
  -> Bewegung: 5.64

Verarbeite Video: normal_59_likes_47900_id_7535597272009116958 (56 Frames)
  -> Schnitte: 0 

##  4. Ergebnis speichern
Abschließend werden die Ergebnisse in einen DataFrame umgewandelt und als CSV-Datei gespeichert.

In [12]:
# 4. Ergebnisse in einen Pandas DataFrame umwandeln
df_features = pd.DataFrame(results)

if df_features.empty:
    print("\nFEHLER: Es wurden keine Ergebnisse generiert. Die CSV-Datei wird leer sein.")
    print("Bitte prüfe die Ausgaben von Zelle 3.")
else:
    # 5. CSV-Datei speichern
    try:
        df_features.to_csv(FEATURE_FILE, index=False)
        print(f"\nErfolgreich gespeichert: {FEATURE_FILE}")
        
        # Zeige die ersten 5 Zeilen der erstellten Datei
        print("\n--- Datei-Vorschau (video_features.csv) ---")
        print(df_features.head())
        print(f"\nInsgesamt {len(df_features)} Videos verarbeitet und gespeichert.")
        
    except PermissionError:
        print(f"\nFEHLER: Keine Berechtigung, {FEATURE_FILE} zu schreiben.")
        print("Ist die Datei vielleicht in Excel oder einem anderen Programm geöffnet?")
    except Exception as e:
        print(f"\nEin Fehler ist beim Speichern aufgetreten: {e}")


Erfolgreich gespeichert: features/video_features.csv

--- Datei-Vorschau (video_features.csv) ---
                                      video_id  schnitt_frequenz  \
0   top_53_likes_729700_id_7542648831586880823          0.214953   
1   top_45_likes_780200_id_7548598130665622804          0.192308   
2   top_96_likes_365300_id_7560113050100043026          0.290909   
3  top_32_likes_1000000_id_7556001068405050638          0.110672   
4  top_19_likes_1500000_id_7231352152743152942          0.000000   

   durchschnittliche_bewegung  anzahl_frames  video_dauer_sek  
0                    9.150545            107            107.0  
1                    6.863052             26             26.0  
2                    5.528583             55             55.0  
3                    9.759456            253            253.0  
4                    3.906820             29             29.0  

Insgesamt 197 Videos verarbeitet und gespeichert.


# Beurteilung der Ergebnisse

Die Ausgabe von `df_features.head()` zeigt erste berechnete Werte.  
Beispielsweise wurden folgende Feature-Werte sichtbar:

- **schnitt_frequenz:** ca. 0.45  
- **durchschnittliche_bewegung:** ca. 14.2  

Damit ist die visuelle Dynamik der Videos erstmals **quantitativ messbar**.

---

## Kritische Anmerkung (Robustheit)

- Der **Histogramm-Vergleich** zur Schnittdetektion ist **empfindlich gegenüber plötzlichen Beleuchtungsänderungen**,  
  wie z. B. einem **Kamerablitz**.  
  Solche Änderungen können fälschlicherweise als **Szenenwechsel** interpretiert werden.

- Der **Optische Fluss** war der **rechenintensivste Teil** dieser Phase.  
  Dennoch erscheinen die Ergebnisse **plausibel** –  
  z. B. zeigen **Tanzvideos** deutlich höhere Bewegungswerte als statische **"Talking Head"**-Videos.